# Zero-Error Algorithm Comparison

This notebook runs the exact dynamic-programming algorithm and the greedy gain-based approximation on the complete 1D and 2D zero-error benchmarks.

It reports the numerical summaries used in the manuscript and displays the three selected publication figures. Existing benchmark inputs are reused by default; set `GENERATE_MISSING_INPUTS = True` only if the generated inputs are absent and must be recreated from the fixed benchmark definitions.


In [7]:
from pathlib import Path
import sys

from IPython.display import Markdown, display

# Resolve paths whether the notebook starts from the repository root or this folder.
working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "min-sensor-zero-error"
    if (working_directory / "min-sensor-zero-error").is_dir()
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
SCRIPT_DIRECTORY = PROJECT_DIRECTORY / "scripts"
for directory in (SOURCE_DIRECTORY, SCRIPT_DIRECTORY):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

# Reload local modules when the notebook is restarted after code changes.
local_modules = (
    "optimal", "greedy", "phase_data", "field_models", "eval",
    "paper_analysis", "generate_inputs", "generate_benchmarks",
    "evaluate_benchmarks",
)
for module_name in local_modules:
    sys.modules.pop(module_name, None)

from evaluate_benchmarks import run_benchmark_evaluation
from paper_analysis import family_excess_table, guarantee_table, heatmap_table

print("Project directory:", PROJECT_DIRECTORY)
print("Source directory:", SOURCE_DIRECTORY)
print("Script directory:", SCRIPT_DIRECTORY)


Project directory: /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error
Source directory: /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/src
Script directory: /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/scripts


## Run Settings

`REPEATS` affects runtime averaging only. `BOOTSTRAP_ITERATIONS` controls confidence intervals. `GENERATE_MISSING_INPUTS = False` keeps the notebook on the saved benchmark inputs and prevents accidental data regeneration.


In [8]:
# Number of deterministic runs used to average each runtime measurement.
REPEATS = 1

# Number of stratified bootstrap samples used for confidence intervals.
BOOTSTRAP_ITERATIONS = 2_000

# Keep saved benchmark inputs fixed by default.
GENERATE_MISSING_INPUTS = False

print("Run mode: all 11,520 saved benchmark cases")
print("Runtime repetitions:", REPEATS)
print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)
print("Generate missing inputs:", GENERATE_MISSING_INPUTS)


Run mode: all 11,520 saved benchmark cases
Runtime repetitions: 1
Bootstrap iterations: 2000
Generate missing inputs: False


## Run Complete Evaluation

This cell evaluates both dimensions and creates the statistical analysis and figures. With the default settings, it requires the existing `benchmark_1d` and `benchmark_2d` inputs and does not regenerate data.


In [9]:
# Evaluate both dimensions and create the statistical analysis and figures.
analysis = run_benchmark_evaluation(
    repeats=REPEATS,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    generate_missing=GENERATE_MISSING_INPUTS,
)

# Stop if the analysis does not contain exactly the selected publication figures.
SELECTED_FIGURES = ("families", "guarantee", "heatmaps")
if set(analysis["figures"]) != set(SELECTED_FIGURES):
    raise RuntimeError("The expected three-figure analysis was not created.")


Evaluating benchmark_1d ...
  5760 cases -> /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/results/algorithm_comparison/benchmark_1d_results_full/tables
Evaluating benchmark_2d ...
  5760 cases -> /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/results/algorithm_comparison/benchmark_2d_results_full/tables
Benchmark tables and figures -> /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/results/algorithm_comparison/results_summary_benchmark_1d_benchmark_2d_full


## Main Numerical Summary

The first row combines both dimensions. The following rows report 1D and 2D separately. The primary quality metric is the greedy-to-optimal sensor ratio, $N_{greedy}/N_{optimal}$.


In [10]:
# Format summary records as one Markdown table.
def summary_table(rows):
    header = (
        "| Group | Cases | Mean ratio | 95% CI | Exact-match rate | "
        "95th percentile | Maximum |\n"
        "|---|---:|---:|---:|---:|---:|---:|"
    )
    body = []
    for row in rows:
        body.append(
            f"| {row['group_value']} | {row['case_count']} | "
            f"{row['mean_greedy_ratio']:.3f} | "
            f"[{row['mean_greedy_ratio_ci95_low']:.3f}, "
            f"{row['mean_greedy_ratio_ci95_high']:.3f}] | "
            f"{100 * row['exact_match_rate']:.1f}% | "
            f"{row['p95_greedy_ratio']:.3f} | "
            f"{row['max_greedy_ratio']:.3f} |"
        )
    return header + "\n" + "\n".join(body)

summary_rows = [analysis["overall_summary"], *analysis["dimension_summaries"]]
display(Markdown(summary_table(summary_rows)))

| Group | Cases | Mean ratio | 95% CI | Exact-match rate | 95th percentile | Maximum |
|---|---:|---:|---:|---:|---:|---:|
| combined | 11520 | 1.152 | [1.149, 1.155] | 30.1% | 1.487 | 2.280 |
| 1D | 5760 | 1.161 | [1.157, 1.165] | 28.6% | 1.500 | 2.280 |
| 2D | 5760 | 1.144 | [1.141, 1.148] | 31.6% | 1.478 | 2.164 |

## Figure Data Tables

These tables contain the numerical values used in the three publication figures. They are computed from the raw per-case results in `analysis["rows"]`, not from image files.


In [11]:
def markdown_table(rows, columns, formats):
    header = "| " + " | ".join(label for label, _ in columns) + " |"
    separator = "|" + "|".join("---:" if formats.get(key) else "---" for _, key in columns) + "|"
    body = []
    for row in rows:
        values = []
        for _, key in columns:
            formatter = formats.get(key)
            value = row[key]
            values.append(formatter(value) if formatter else str(value))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *body])

heatmap_rows = heatmap_table(analysis["rows"])
display(Markdown("### Quality Heatmap Values"))
display(Markdown(markdown_table(
    heatmap_rows,
    (
        ("Dimension", "dimension"),
        ("K", "K"),
        ("M", "M"),
        ("Mean additional sensors used by Greedy (%)", "mean_additional_sensors_percent"),
    ),
    {"K": lambda value: f"{int(value)}", "M": lambda value: f"{int(value)}", "mean_additional_sensors_percent": lambda value: f"{float(value):.2f}"},
)))

family_rows = family_excess_table(
    analysis["rows"],
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)
display(Markdown("### Field-Family Bar-Chart Values"))
display(Markdown(markdown_table(
    family_rows,
    (
        ("Dimension", "dimension"),
        ("Field family", "field_family"),
        ("K", "K"),
        ("Mean additional sensors used by Greedy (%)", "mean_additional_sensors_percent"),
        ("95% CI low", "ci95_low"),
        ("95% CI high", "ci95_high"),
    ),
    {
        "K": lambda value: f"{int(value)}",
        "mean_additional_sensors_percent": lambda value: f"{float(value):.2f}",
        "ci95_low": lambda value: f"{float(value):.2f}",
        "ci95_high": lambda value: f"{float(value):.2f}",
    },
)))

guarantee_rows = guarantee_table(analysis["rows"])
display(Markdown("### Approximation-Guarantee Figure Values"))
display(Markdown(markdown_table(
    guarantee_rows,
    (
        ("Dimension", "dimension"),
        ("Field family", "field_family"),
        ("K", "K"),
        ("Maximum worst-case allowance used (%)", "maximum_worst_case_allowance_used_percent"),
    ),
    {"K": lambda value: f"{int(value)}", "maximum_worst_case_allowance_used_percent": lambda value: f"{float(value):.2f}"},
)))


### Quality Heatmap Values

| Dimension | K | M | Mean additional sensors used by Greedy (%) |
|---|---:|---:|---:|
| 1D | 3 | 3 | 8.22 |
| 1D | 3 | 4 | 7.30 |
| 1D | 3 | 5 | 7.99 |
| 1D | 3 | 6 | 11.43 |
| 1D | 3 | 7 | 13.58 |
| 1D | 3 | 8 | 8.64 |
| 1D | 3 | 9 | 13.35 |
| 1D | 3 | 10 | 11.75 |
| 1D | 4 | 3 | 14.14 |
| 1D | 4 | 4 | 12.51 |
| 1D | 4 | 5 | 17.03 |
| 1D | 4 | 6 | 15.31 |
| 1D | 4 | 7 | 13.78 |
| 1D | 4 | 8 | 12.52 |
| 1D | 4 | 9 | 19.13 |
| 1D | 4 | 10 | 15.06 |
| 1D | 5 | 3 | 12.51 |
| 1D | 5 | 4 | 14.21 |
| 1D | 5 | 5 | 16.24 |
| 1D | 5 | 6 | 13.82 |
| 1D | 5 | 7 | 14.06 |
| 1D | 5 | 8 | 16.87 |
| 1D | 5 | 9 | 17.75 |
| 1D | 5 | 10 | 17.56 |
| 1D | 6 | 3 | 10.04 |
| 1D | 6 | 4 | 12.60 |
| 1D | 6 | 5 | 15.88 |
| 1D | 6 | 6 | 18.52 |
| 1D | 6 | 7 | 15.76 |
| 1D | 6 | 8 | 13.96 |
| 1D | 6 | 9 | 23.57 |
| 1D | 6 | 10 | 22.32 |
| 1D | 7 | 3 | 10.72 |
| 1D | 7 | 4 | 13.69 |
| 1D | 7 | 5 | 18.71 |
| 1D | 7 | 6 | 15.90 |
| 1D | 7 | 7 | 17.36 |
| 1D | 7 | 8 | 19.78 |
| 1D | 7 | 9 | 18.20 |
| 1D | 7 | 10 | 21.36 |
| 1D | 8 | 3 | 11.15 |
| 1D | 8 | 4 | 15.13 |
| 1D | 8 | 5 | 16.41 |
| 1D | 8 | 6 | 15.65 |
| 1D | 8 | 7 | 18.53 |
| 1D | 8 | 8 | 21.34 |
| 1D | 8 | 9 | 21.02 |
| 1D | 8 | 10 | 19.69 |
| 1D | 9 | 3 | 11.70 |
| 1D | 9 | 4 | 14.46 |
| 1D | 9 | 5 | 17.66 |
| 1D | 9 | 6 | 18.08 |
| 1D | 9 | 7 | 21.53 |
| 1D | 9 | 8 | 17.88 |
| 1D | 9 | 9 | 21.34 |
| 1D | 9 | 10 | 23.71 |
| 1D | 10 | 3 | 11.91 |
| 1D | 10 | 4 | 13.30 |
| 1D | 10 | 5 | 19.70 |
| 1D | 10 | 6 | 17.09 |
| 1D | 10 | 7 | 20.93 |
| 1D | 10 | 8 | 20.70 |
| 1D | 10 | 9 | 22.94 |
| 1D | 10 | 10 | 22.91 |
| 2D | 3 | 3 | 10.42 |
| 2D | 3 | 4 | 9.95 |
| 2D | 3 | 5 | 10.38 |
| 2D | 3 | 6 | 11.48 |
| 2D | 3 | 7 | 10.34 |
| 2D | 3 | 8 | 11.53 |
| 2D | 3 | 9 | 12.34 |
| 2D | 3 | 10 | 9.90 |
| 2D | 4 | 3 | 9.67 |
| 2D | 4 | 4 | 14.24 |
| 2D | 4 | 5 | 12.94 |
| 2D | 4 | 6 | 12.42 |
| 2D | 4 | 7 | 10.86 |
| 2D | 4 | 8 | 12.58 |
| 2D | 4 | 9 | 11.52 |
| 2D | 4 | 10 | 11.62 |
| 2D | 5 | 3 | 8.77 |
| 2D | 5 | 4 | 14.25 |
| 2D | 5 | 5 | 13.05 |
| 2D | 5 | 6 | 15.86 |
| 2D | 5 | 7 | 16.06 |
| 2D | 5 | 8 | 16.29 |
| 2D | 5 | 9 | 13.81 |
| 2D | 5 | 10 | 15.98 |
| 2D | 6 | 3 | 10.27 |
| 2D | 6 | 4 | 12.86 |
| 2D | 6 | 5 | 14.37 |
| 2D | 6 | 6 | 16.62 |
| 2D | 6 | 7 | 16.18 |
| 2D | 6 | 8 | 17.32 |
| 2D | 6 | 9 | 14.68 |
| 2D | 6 | 10 | 16.52 |
| 2D | 7 | 3 | 11.43 |
| 2D | 7 | 4 | 12.41 |
| 2D | 7 | 5 | 14.85 |
| 2D | 7 | 6 | 13.56 |
| 2D | 7 | 7 | 16.30 |
| 2D | 7 | 8 | 17.89 |
| 2D | 7 | 9 | 20.87 |
| 2D | 7 | 10 | 16.58 |
| 2D | 8 | 3 | 12.18 |
| 2D | 8 | 4 | 11.90 |
| 2D | 8 | 5 | 14.44 |
| 2D | 8 | 6 | 18.95 |
| 2D | 8 | 7 | 14.54 |
| 2D | 8 | 8 | 18.38 |
| 2D | 8 | 9 | 17.32 |
| 2D | 8 | 10 | 16.35 |
| 2D | 9 | 3 | 12.44 |
| 2D | 9 | 4 | 14.35 |
| 2D | 9 | 5 | 13.70 |
| 2D | 9 | 6 | 14.99 |
| 2D | 9 | 7 | 17.79 |
| 2D | 9 | 8 | 17.14 |
| 2D | 9 | 9 | 17.32 |
| 2D | 9 | 10 | 20.45 |
| 2D | 10 | 3 | 11.68 |
| 2D | 10 | 4 | 13.90 |
| 2D | 10 | 5 | 15.65 |
| 2D | 10 | 6 | 16.76 |
| 2D | 10 | 7 | 17.78 |
| 2D | 10 | 8 | 16.44 |
| 2D | 10 | 9 | 18.65 |
| 2D | 10 | 10 | 20.81 |

### Field-Family Bar-Chart Values

| Dimension | Field family | K | Mean additional sensors used by Greedy (%) | 95% CI low | 95% CI high |
|---|---|---:|---:|---:|---:|
| 1D | gaussian | 3 | 9.89 | 8.11 | 11.84 |
| 1D | gaussian | 4 | 14.55 | 12.24 | 16.86 |
| 1D | gaussian | 5 | 15.59 | 13.43 | 17.90 |
| 1D | gaussian | 6 | 15.67 | 13.66 | 17.69 |
| 1D | gaussian | 7 | 18.06 | 16.28 | 19.92 |
| 1D | gaussian | 8 | 18.12 | 16.19 | 19.98 |
| 1D | gaussian | 9 | 18.42 | 16.30 | 20.59 |
| 1D | gaussian | 10 | 18.27 | 16.46 | 20.23 |
| 1D | sinusoidal | 3 | 8.58 | 6.87 | 10.33 |
| 1D | sinusoidal | 4 | 14.22 | 12.01 | 16.45 |
| 1D | sinusoidal | 5 | 14.55 | 12.61 | 16.61 |
| 1D | sinusoidal | 6 | 16.50 | 14.61 | 18.55 |
| 1D | sinusoidal | 7 | 17.44 | 15.48 | 19.46 |
| 1D | sinusoidal | 8 | 19.07 | 17.37 | 20.81 |
| 1D | sinusoidal | 9 | 19.60 | 17.55 | 21.65 |
| 1D | sinusoidal | 10 | 22.48 | 20.60 | 24.49 |
| 1D | quadratic | 3 | 12.38 | 10.12 | 14.54 |
| 1D | quadratic | 4 | 16.03 | 13.85 | 18.35 |
| 1D | quadratic | 5 | 16.00 | 13.90 | 18.43 |
| 1D | quadratic | 6 | 17.58 | 15.19 | 19.82 |
| 1D | quadratic | 7 | 15.39 | 13.57 | 17.26 |
| 1D | quadratic | 8 | 14.91 | 13.12 | 16.84 |
| 1D | quadratic | 9 | 16.87 | 14.81 | 18.96 |
| 1D | quadratic | 10 | 15.30 | 13.24 | 17.27 |
| 2D | gaussian | 3 | 5.73 | 4.19 | 7.44 |
| 2D | gaussian | 4 | 6.35 | 5.01 | 7.87 |
| 2D | gaussian | 5 | 9.10 | 7.64 | 10.64 |
| 2D | gaussian | 6 | 10.04 | 8.64 | 11.61 |
| 2D | gaussian | 7 | 10.32 | 8.99 | 11.77 |
| 2D | gaussian | 8 | 9.47 | 8.26 | 10.71 |
| 2D | gaussian | 9 | 10.51 | 9.29 | 11.91 |
| 2D | gaussian | 10 | 10.70 | 9.48 | 12.02 |
| 2D | sinusoidal | 3 | 7.86 | 5.96 | 9.93 |
| 2D | sinusoidal | 4 | 9.13 | 7.30 | 11.05 |
| 2D | sinusoidal | 5 | 12.28 | 10.39 | 14.35 |
| 2D | sinusoidal | 6 | 12.73 | 11.07 | 14.45 |
| 2D | sinusoidal | 7 | 12.37 | 10.89 | 13.94 |
| 2D | sinusoidal | 8 | 15.02 | 13.30 | 16.80 |
| 2D | sinusoidal | 9 | 15.84 | 14.28 | 17.42 |
| 2D | sinusoidal | 10 | 15.81 | 14.25 | 17.54 |
| 2D | quadratic | 3 | 18.79 | 16.61 | 21.09 |
| 2D | quadratic | 4 | 20.46 | 18.15 | 22.80 |
| 2D | quadratic | 5 | 21.39 | 18.83 | 23.90 |
| 2D | quadratic | 6 | 21.78 | 19.02 | 24.51 |
| 2D | quadratic | 7 | 23.77 | 21.15 | 26.43 |
| 2D | quadratic | 8 | 22.04 | 19.47 | 24.63 |
| 2D | quadratic | 9 | 21.72 | 19.54 | 23.90 |
| 2D | quadratic | 10 | 22.87 | 20.32 | 25.43 |

### Approximation-Guarantee Figure Values

| Dimension | Field family | K | Maximum worst-case allowance used (%) |
|---|---|---:|---:|
| 1D | gaussian | 3 | 60.68 |
| 1D | gaussian | 4 | 50.23 |
| 1D | gaussian | 5 | 38.00 |
| 1D | gaussian | 6 | 31.89 |
| 1D | gaussian | 7 | 26.87 |
| 1D | gaussian | 8 | 24.38 |
| 1D | gaussian | 9 | 20.79 |
| 1D | gaussian | 10 | 24.42 |
| 1D | sinusoidal | 3 | 60.68 |
| 1D | sinusoidal | 4 | 41.86 |
| 1D | sinusoidal | 5 | 43.43 |
| 1D | sinusoidal | 6 | 29.01 |
| 1D | sinusoidal | 7 | 28.47 |
| 1D | sinusoidal | 8 | 21.18 |
| 1D | sinusoidal | 9 | 21.70 |
| 1D | sinusoidal | 10 | 21.89 |
| 1D | quadratic | 3 | 60.68 |
| 1D | quadratic | 4 | 51.82 |
| 1D | quadratic | 5 | 40.33 |
| 1D | quadratic | 6 | 47.27 |
| 1D | quadratic | 7 | 23.22 |
| 1D | quadratic | 8 | 23.03 |
| 1D | quadratic | 9 | 23.48 |
| 1D | quadratic | 10 | 19.50 |
| 2D | gaussian | 3 | 60.68 |
| 2D | gaussian | 4 | 40.79 |
| 2D | gaussian | 5 | 29.71 |
| 2D | gaussian | 6 | 22.40 |
| 2D | gaussian | 7 | 19.28 |
| 2D | gaussian | 8 | 15.29 |
| 2D | gaussian | 9 | 18.86 |
| 2D | gaussian | 10 | 18.19 |
| 2D | sinusoidal | 3 | 60.68 |
| 2D | sinusoidal | 4 | 55.81 |
| 2D | sinusoidal | 5 | 36.19 |
| 2D | sinusoidal | 6 | 20.51 |
| 2D | sinusoidal | 7 | 18.25 |
| 2D | sinusoidal | 8 | 18.01 |
| 2D | sinusoidal | 9 | 19.09 |
| 2D | sinusoidal | 10 | 17.51 |
| 2D | quadratic | 3 | 60.68 |
| 2D | quadratic | 4 | 46.51 |
| 2D | quadratic | 5 | 40.43 |
| 2D | quadratic | 6 | 38.69 |
| 2D | quadratic | 7 | 37.74 |
| 2D | quadratic | 8 | 28.47 |
| 2D | quadratic | 9 | 22.64 |
| 2D | quadratic | 10 | 30.57 |

## Selected Publication Figures

1. **Sensor excess by field family:** compares Gaussian, sinusoidal, and quadratic fields across $K$.
2. **Approximation guarantee comparison:** shows how much of the theoretical sensor-excess allowance was used.
3. **Quality heatmaps:** shows mean Greedy sensor excess across $K$ and $M$.

All figure text is formatted with Times New Roman, 12 pt.


In [12]:
# Display only the three selected figures in the requested order.
plot_titles = {
    "families": "Sensor excess by field family",
    "guarantee": "Approximation guarantee comparison",
    "heatmaps": "Quality heatmaps",
}
for figure_key in SELECTED_FIGURES:
    display(Markdown(f"### {plot_titles[figure_key]}"))
    display(analysis["figures"][figure_key])


### Sensor excess by field family

<Figure size 1150x440 with 2 Axes>

### Approximation guarantee comparison

<Figure size 1150x420 with 2 Axes>

### Quality heatmaps

<Figure size 1150x480 with 3 Axes>

## Saved Files

Detailed result tables are saved under `results/algorithm_comparison/<dataset>_results_full/tables/`. Combined analysis tables and the three PNG/PDF figure pairs are saved under `results/algorithm_comparison/results_summary_benchmark_1d_benchmark_2d_full/`.
